# 05 — Analyse Structurelle : Mismatch Éducation-Emploi

### Objectif de ce notebook
Répondre à la question centrale du projet :

> **Le marché du travail ivoirien est-il capable d'absorber
> les jeunes éduqués qu'il forme ?**

**Trois analyses réalisées :**
1. Vulnérabilité par niveau d'éducation — confirmation du paradoxe
2. Structure sectorielle — concentration et fragilité
3. Mismatch index — mesure du décalage formel/informel

In [33]:
import pandas as pd

In [34]:
# LOAD DATA
df_edu = pd.read_csv('../data/processed/ci_education_15_35.csv')
df_sector = pd.read_csv('../data/processed/ci_sector_15_35.csv')

### 1. Vulnérabilité par niveau d'éducation
Calcul de l'indice de vulnérabilité moyen pour chaque
niveau d'éducation sur toute la période 2015–2025.

> `vulnerability = share_unemp + share_inact`

In [35]:
# EDUCATION VULNERABILITY ANALYSIS

df_edu['vulnerability'] = (
    df_edu['share_unemp'] + df_edu['share_inact']
) 

edu_summary = (
    df_edu.groupby('edu_ilo')['vulnerability']
    .mean()
    .reset_index()
)

print("\n=== VULNERABILITY BY EDUCATION LEVEL ===")
print(edu_summary)


=== VULNERABILITY BY EDUCATION LEVEL ===
             edu_ilo  vulnerability
0  Less than primary       0.309584
1    Lower secondary       0.510649
2            Primary       0.445038
3           Tertiary       0.624780
4    Upper secondary       0.599032


### 2. Structure sectorielle de l'emploi
Identification de la concentration de l'emploi
par secteur base du calcul du mismatch.

In [36]:
# EMPLOYMENT STRUCTURE BY SECTOR

sector_summary = (
    df_sector.groupby('sector21')['population']
    .sum()
    .reset_index()
)

sector_summary['share'] = (
    sector_summary['population'] / sector_summary['population'].sum()
)

print("\n=== TOP EMPLOYMENT SECTORS ===")
print(
    sector_summary
    .sort_values('share', ascending=False)
    .head(10)
)


=== TOP EMPLOYMENT SECTORS ===
                                             sector21  population     share
4                   Agriculture; forestry and fishing  28897468.0  0.486310
20  Wholesale and retail trade; repair of motor ve...  11249038.0  0.189308
12                                      Manufacturing   4804207.0  0.080849
0           Accommodation and food service activities   2692696.0  0.045315
14                           Other service activities   2369389.0  0.039874
18                         Transportation and storage   2016180.0  0.033930
6                                        Construction   1650030.0  0.027768
2   Activities of households as employers; undiffe...   1244720.0  0.020947
7                                           Education    910465.0  0.015322
16  Public administration and defence; compulsory ...    834494.0  0.014044


### 3. Concentration Agriculture + Commerce
Ces deux secteurs sont majoritairement **informels**
et à **faible valeur ajoutée**.
Une concentration supérieure à 60% signale
une économie structurellement fragile.

In [37]:
# FINAL INDICATOR

agriculture_share = sector_summary[
    sector_summary['sector21'].str.contains('Agriculture', case=False, na=False)
]['share'].sum()

commerce_share = sector_summary[
    sector_summary['sector21'].str.contains('trade|retail|commerce', case=False, na=False)
]['share'].sum()

total_exposure = agriculture_share + commerce_share

print("\n=== FINAL INSIGHT ===")
print(f"Agriculture + Commerce Exposure: {total_exposure:.2f}")

if total_exposure > 0.6:
    print("High concentration of youth in vulnerable sectors")
else:
    print("More diversified economic structure")


=== FINAL INSIGHT ===
Agriculture + Commerce Exposure: 0.68
High concentration of youth in vulnerable sectors


### 4. Interprétation structurelle
Vérification croisée de la concentration sectorielle
pour confirmer la fragilité du marché du travail.

In [38]:
# AUTOMATED INSIGHT INTERPRETATION

agri = sector_summary[
    sector_summary['sector21'].str.contains('Agriculture', case=False, na=False)
]['share'].sum()

commerce = sector_summary[
    sector_summary['sector21'].str.contains('trade|retail|commerce', case=False, na=False)
]['share'].sum()

total_risk_concentration = agri + commerce

print("\n=== STRUCTURAL INSIGHT ===")
print(f"Agriculture + Commerce Concentration: {round(total_risk_concentration, 2)}")

if total_risk_concentration > 0.6:
    print("High dependence on low-quality employment sectors")
else:
    print("More diversified labor market structure")


=== STRUCTURAL INSIGHT ===
Agriculture + Commerce Concentration: 0.68
High dependence on low-quality employment sectors


### 5. Mismatch Index — Formel vs Informel

**Logique du calcul :**
Les secteurs sont classifiés en trois catégories :

| Catégorie | Exemples |
|---|---|
| **Informel** | Agriculture, Construction, Ménages |
| **Formel** | Éducation, Administration, Finance, Santé |
| **Mixte** | Commerce, Manufacture, Transport |

Un mismatch élevé signifie que le secteur formel
est trop étroit pour absorber les jeunes éduqués
qui arrivent sur le marché du travail.

In [39]:
# MISMATCH INDEX — VERSION CORRIGÉE
# Logique : compare la part de l'emploi FORMEL vs INFORMEL
# Un marché formel trop étroit = mismatch pour les jeunes éduqués

informal_keywords = ['Agriculture', 'household', 'Construction', 'undiffe']
formal_keywords = ['Education', 'administration', 'Financial',
                   'Health', 'Professional', 'Information']

def classify_sector(s):
    s = s.lower()
    if any(k.lower() in s for k in informal_keywords):
        return 'Informel'
    elif any(k.lower() in s for k in formal_keywords):
        return 'Formel'
    else:
        return 'Mixte'

sector_summary['sector_type'] = sector_summary['sector21'].apply(classify_sector)

formal_share = sector_summary[
    sector_summary['sector_type'] == 'Formel'
]['share'].sum()

informal_share = sector_summary[
    sector_summary['sector_type'] == 'Informel'
]['share'].sum()

mismatch_score = informal_share - formal_share

print(f"Emploi informel  : {informal_share:.1%}")
print(f"Emploi formel    : {formal_share:.1%}")
print(f"Mismatch score   : {mismatch_score:.1%}")

if mismatch_score > 0.4:
    print(" Marché formel trop étroit pour absorber les jeunes éduqués")

Emploi informel  : 53.5%
Emploi formel    : 5.6%
Mismatch score   : 47.9%
 Marché formel trop étroit pour absorber les jeunes éduqués


### 6. Conclusion analytique

Les trois résultats convergent vers un même diagnostic :

In [40]:
# FINAL CONCLUSION
print("\n=== CONCLUSION ===")
if mismatch_score > 0.4:
    print(" Mismatch critique : le secteur formel est trop étroit")
    print("   pour absorber les jeunes éduqués de Côte d'Ivoire.")
else:
    print("Mismatch modéré — diversification en cours.")


=== CONCLUSION ===
 Mismatch critique : le secteur formel est trop étroit
   pour absorber les jeunes éduqués de Côte d'Ivoire.


### Synthèse finale

| Constat | Chiffre clé | Implication politique |
|---|---|---|
| Paradoxe du diplômé | Vulnérabilité tertiaire = la plus haute | Réformer les filières universitaires |
| Concentration informelle | 68% dans agriculture + commerce | Diversifier l'économie |
| Mismatch structurel | Secteur formel < 15% de l'emploi | Élargir le secteur formel |

> **Le problème de l'emploi des jeunes en Côte d'Ivoire
> n'est pas un manque de diplômés  c'est un marché
> formel trop étroit pour les accueillir.**